In [19]:

import pandas as pd
import numpy as np
import glob
import os


In [20]:
import sys
print(sys.executable)

c:\codes\btp\venv32\Scripts\python.exe


In [21]:
import sys
!{sys.executable} -m pip install efficient-kan

In [22]:

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, matthews_corrcoef
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings("ignore")

In [23]:
DATA_PATH = "C:/codes/btp_datasets/TestCloudIDS_Dataset"

all_files = glob.glob(os.path.join(DATA_PATH, "**/*.csv"), recursive=True)

df_list = []
for file in all_files:
    temp = pd.read_csv(file)
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

print("Total shape:", df.shape)
df.tail()


Total shape: (119933, 84)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
119928,192.168.136.128-192.168.136.129-1145-38626-6,192.168.136.129,38626,192.168.136.128,1145,6,23/01/2024 12:30:25 PM,47,0,2,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RS_Scanner
119929,192.168.136.128-192.168.136.129-21-46376-6,192.168.136.129,46376,192.168.136.128,21,6,23/01/2024 12:30:24 PM,2874,2,2,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RS_Scanner
119930,192.168.136.128-192.168.136.129-21-46376-6,192.168.136.128,21,192.168.136.129,46376,6,23/01/2024 12:30:24 PM,25428,2,1,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
119931,192.168.136.128-192.168.136.129-81-47188-6,192.168.136.129,47188,192.168.136.128,81,6,23/01/2024 12:30:24 PM,1266,2,2,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RS_Scanner
119932,192.168.136.128-192.168.136.129-443-38532-6,192.168.136.129,38532,192.168.136.128,443,6,23/01/2024 12:30:24 PM,857,2,2,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RS_Scanner


In [6]:

if "Timestamp" in df.columns:
    df = df.drop(columns=["Timestamp"])

# Replace infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows with missing labels
df = df.dropna(subset=["Label"])

In [7]:
from sklearn.preprocessing import LabelEncoder

min_samples = 50
class_counts = df["Label"].value_counts()
valid_classes = class_counts[class_counts >= min_samples].index
df = df[df["Label"].isin(valid_classes)]

le = LabelEncoder()
df["Label"] = le.fit_transform(df["Label"])
print("Classes:", le.classes_)

Classes: ['Benign' 'DDoS_Ripper' 'GoldenEye' 'HOIC' 'Hulk' 'LOIC_HTTP' 'RS_Scanner'
 'RS_l7_HTTP' 'SYN_Flood' 'TorsHammer' 'Xerxes']


In [8]:
X = df.drop(columns=["Label"])
y = df["Label"]
cols_to_drop = []
X = X.select_dtypes(include=['number'])
print("Dropping non-numeric columns:", cols_to_drop)
X = X.drop(columns=cols_to_drop)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)


y_train = y_train.astype(int)
y_test = y_test.astype(int)

imputer = SimpleImputer(strategy="mean")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Imputation Done")

feature_names = X_train.columns.tolist()
print("Label Distribution:\n", y.value_counts())

Dropping non-numeric columns: []
Train Shape: (95913, 79)
Test Shape: (23979, 79)
Imputation Done
Label Distribution:
 Label
7     28235
0     25999
8     20909
5     10867
1     10020
3      8883
2      7407
4      2024
9      2024
10     2024
6      1500
Name: count, dtype: int64


In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from efficient_kan import KAN

from sklearn.metrics import classification_report

In [11]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test.values,  dtype=torch.long)

In [12]:
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=128, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [13]:
num_classes = len(np.unique(y_train))
num_features = X_train.shape[1]

kan_model = KAN(
    layers_hidden=[num_features, 32, num_classes]
).to(device)

print(kan_model)

KAN(
  (layers): ModuleList(
    (0-1): 2 x KANLinear(
      (base_activation): SiLU()
    )
  )
)


In [14]:
# CLASS WEIGHTS FOR IMBALANCE
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = torch.optim.Adam(kan_model.parameters(), lr=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

In [15]:
epochs = 20
best_val_loss = float('inf')
patience_counter = 0
patience = 15
X_test_t  = X_test_t.to(device)
y_test_t  = y_test_t.to(device)


In [16]:
for epoch in range(epochs):
    kan_model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = kan_model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # Track training accuracy
        preds = output.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    train_acc  = correct / total
    train_loss = total_loss / len(train_loader)

    # Validation
    kan_model.eval()
    with torch.no_grad():
        val_output = kan_model(X_test_t)
        val_loss   = criterion(val_output, y_test_t).item()
        val_preds  = val_output.argmax(dim=1)
        val_acc    = (val_preds == y_test_t).float().mean().item()

    scheduler.step(val_loss)
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(kan_model.state_dict(), "best_kan.pt")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

Epoch 01 | Train Loss: 0.4184 | Train Acc: 0.9212 | Val Loss: 0.3582 | Val Acc: 0.9481
Epoch 02 | Train Loss: 0.3525 | Train Acc: 0.9414 | Val Loss: 0.3819 | Val Acc: 0.9438
Epoch 03 | Train Loss: 0.3490 | Train Acc: 0.9442 | Val Loss: 0.3866 | Val Acc: 0.9438
Epoch 04 | Train Loss: 0.3472 | Train Acc: 0.9465 | Val Loss: 0.3543 | Val Acc: 0.9397
Epoch 05 | Train Loss: 0.3425 | Train Acc: 0.9478 | Val Loss: 0.3910 | Val Acc: 0.8897
Epoch 06 | Train Loss: 0.3394 | Train Acc: 0.9491 | Val Loss: 0.3374 | Val Acc: 0.9550
Epoch 07 | Train Loss: 0.3369 | Train Acc: 0.9483 | Val Loss: 0.3374 | Val Acc: 0.9488
Epoch 08 | Train Loss: 0.3381 | Train Acc: 0.9491 | Val Loss: 0.3439 | Val Acc: 0.9490
Epoch 09 | Train Loss: 0.3360 | Train Acc: 0.9494 | Val Loss: 0.3278 | Val Acc: 0.9575
Epoch 10 | Train Loss: 0.3309 | Train Acc: 0.9520 | Val Loss: 0.3390 | Val Acc: 0.9545
Epoch 11 | Train Loss: 0.3289 | Train Acc: 0.9517 | Val Loss: 0.3298 | Val Acc: 0.9432
Epoch 12 | Train Loss: 0.3251 | Train Acc: 

In [17]:
#EVALUATE KAN 
kan_model.load_state_dict(torch.load("best_kan.pt"))
kan_model.eval()

with torch.no_grad():
    y_pred_kan = kan_model(X_test_t).argmax(dim=1).cpu().numpy()

print("=== KAN Classification Report ===")
print(classification_report(y_test, y_pred_kan, target_names=le.classes_))

=== KAN Classification Report ===
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00      5200
 DDoS_Ripper       0.99      0.94      0.97      2004
   GoldenEye       1.00      1.00      1.00      1481
        HOIC       1.00      1.00      1.00      1777
        Hulk       0.33      0.93      0.49       405
   LOIC_HTTP       1.00      1.00      1.00      2173
  RS_Scanner       1.00      1.00      1.00       300
  RS_l7_HTTP       0.98      1.00      0.99      5647
   SYN_Flood       1.00      1.00      1.00      4182
  TorsHammer       0.34      0.07      0.12       405
      Xerxes       0.00      0.00      0.00       405

    accuracy                           0.96     23979
   macro avg       0.79      0.81      0.78     23979
weighted avg       0.96      0.96      0.95     23979



In [18]:
kan_params = sum(p.numel() for p in kan_model.parameters())
print("KAN Parameters:", kan_params)

KAN Parameters: 28800
